# FGI Data Import
Builds `data/staged/FGI_liveState.xlsx` from the raw import workbooks. 


In [31]:
## LIBRARY IMPORTS
from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


## 1. Local Paths
Resolve the repository from the current working directory and define raw workbook inputs plus the generated live-state output.


In [32]:
# FILEPATHS FOR DATA IMPORT
rootpath = Path.cwd()
if not (rootpath / 'data' / 'raw').exists():
    raise FileNotFoundError(f'Raw data directory not found under {rootpath}')

filepaths = {
    'FAstatus': rootpath / 'data' / 'raw' / 'FA_Status_FGI_Handoff.xlsx',
    'FGI_Locations': rootpath / 'data' / 'raw' / 'FGI_Locations_wPriority.xlsx',
    'Nodes': rootpath / 'data' / 'raw' / 'Nodes.xlsx',
    'Move Times': rootpath / 'data' / 'staged' / 'move_times' / 'move_time_estimation.xlsx',
    'Paint Schedule': rootpath / 'data' / 'raw' / 'paint_schedules.xlsx'
}

LIVE_STATE_FILE = rootpath / 'data' / 'staged' / 'FGI_liveState.xlsx'


## 2. Data Import Defaults
Defines labor defaults, CPJ values, centerline dependencies, and BTG-to-team mappings exported into live state or used to shape imported data.


In [33]:
## DATA IMPORT DEFAULTS

FGI_STAFFING_BYSHIFT = { # FGI team: # manhours / 8hr shift
            1: {'structure': 16, 'systems': 60, 'declam': 16, 'test': 18},
            2: {'structure': 6, 'systems': 14, 'declam': 4, 'test': 0},
            3: {'structure': 0, 'systems': 0, 'declam': 0, 'test': 0}
        }

FGI_CPJ = { # FGI team: # manhours consumed / 1 BTG completed
        'structure': 6,
        'systems': 3.5,
        'declam': 3,
        'test': 8
    }

CENTERLINES = {
    'A1': None,
    'A2': None,
    'A3': None,
    'A4': None,
    'A5': None,
    'A6': None,
    'A7': None,
    'A8': None,
    'A9': ['C1'],
    'A10': ['C1', 'C2'],
    'BSC1': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'BSC2': ['C1', 'C2', 'C3', 'C3.5', 'C4', 'C5'],
    'C1': None,
    'C2': ['C1'],
    'C3': ['C1', 'C2'],
    'C3.5': ['C1', 'C2', 'C3'],
    'C4': ['C1', 'C2', 'C3', 'C3.5'],
    'C5': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'CR1': ['CR3', 'CR2'],
    'CR2': ['CR3'],
    'CR3': None,
    'D1': None,
    'D2': None,
    'F1': ['C1', 'C2'],
    'F2': ['C1', 'C2'],
    'L4': None,
    'L5': ['L4'],
    'Spur': None,
    'T1': None,
    'T2': None,
    'T3': None,
    'T4': None
}

FGI_TEAMS = ['structure', 'systems', 'declam', 'test']
BTG_TYPES = ['tot', 'p0', 'p1', 'p2', 'p3', 'engines', 'doors', 'test']

BTG_TYPES_BY_LABOR = { # relationships ONLY, NOT 1-to-1 btg conversion
    'structure': ['tot'],
    'systems': ['p2'],
    'declam': ['p3', 'engines'],
    'test': ['test']
}
FGI_TEAMS_BY_BTG_TYPE = { # relationships ONLY, NOT 1-to-1 btg conversion
    'tot': ['structure', 'systems', 'declam', 'test'],
    'FGI_tot': ['structure'],
    'p2': ['systems'],
    'p3': ['declam'],
    'engines': ['declam'],
    'test': ['test']
}
for type in BTG_TYPES:
    if type not in FGI_TEAMS_BY_BTG_TYPE.keys():
        FGI_TEAMS_BY_BTG_TYPE[type] = None



## 3. Generated Live-State Data
FARO, location, labor, move-time, and paint-schedule inputs are cleaned and exported to `data/staged/FGI_liveState.xlsx` with the same workbook styling used by the scheduler trace export.


In [34]:
# dataframe builder functions
def merge_APdata(faro_scorecard, p3_milestones, tankClosure):
    ## organize milestone/tankClosure by LN
    tank_lookup = (
        tankClosure[['LINE_NUMBER', 'TankStatus']]
        .drop_duplicates(subset='LINE_NUMBER')
        .rename(columns={'LINE_NUMBER': 'LN'})
    )
    p3_lookup = (
        p3_milestones
        .rename(columns={'P': 'LN'})
        .copy()
    )

    faro_scorecard = faro_scorecard.copy()
    faro_scorecard['LN'] = pd.to_numeric(faro_scorecard['LN'], errors='coerce')

    tank_lookup['LN'] = pd.to_numeric(tank_lookup['LN'], errors='coerce')
    p3_lookup['LN'] = pd.to_numeric(p3_lookup['LN'], errors='coerce')

    ## store LN as str
    faro_scorecard['LN'] = faro_scorecard['LN'].astype(str)
    tank_lookup['LN'] = tank_lookup['LN'].astype(str)
    p3_lookup['LN'] = p3_lookup['LN'].astype(str)

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    return ap_df

def build_ap_df(faro_scorecard, p3_milestones, tankClosure):
    ap_data = merge_APdata(faro_scorecard, p3_milestones, tankClosure)

    rows = []

    for _, row in ap_data.iterrows():
        ln = int(row['LN'])

        rows.append({
            'LN': ln,
            'FA RO': row['FA RO'],
            'FA RO to B1R': row['FA RO to B1R'],
            'Total Counters': row['Total Counters'],
            'TankStatus': row['TankStatus'],
            'Ceilings': row['Ceilings'],
            'Initial Tests Run': row['Initial Tests Run'],

            # BTG attributes
            'BTG_tot': row['Total BTG'],
            'BTG_p0': row['P0 BTG'],
            'BTG_p1': row['P1 BTG'],
            'BTG_p2': row['P2 BTG'],
            'BTG_p3': row['P3 BTG'],
            'BTG_engines': row['Engines BTG'],
            'BTG_doors': row['Doors BTG'],
            'BTG_test': row['Test BTG'],

            # P3 milestone attributes
            'P3_Engine Hang': row['Engine Hang'],
            'P3_Flight Controls': row['Flight Controls'],
            'P3_Gear Swing': row['Gear Swing'],
            'P3_Medium Pressure Test': row['Medium Pressure Test'],
            'P3_Oil On': row['Oil On'],
            'P3_Power On': row['Power On'],
            'P3_Engine_Type': row['Engine_Type'],
            'P3_Milestone_Completion_Percentage': row['Milestone_Completion_Percentage'],

            # Shake attributes
            'shakes_complete': row['shakes_completed'],
            'shakes_req': row['shakes_req']
        })

    return pd.DataFrame(rows)

def build_location_df(fa_priority, centerlines):
    rows = []
    seen_locations = set()

    for _, row in fa_priority.iterrows():
        loc = row['Location']
        seen_locations.add(loc)
        centerline_deps = centerlines.get(loc, None)
        

        rows.append({
            'Location': loc,
            'Future State Priority': row['Future State Priority'],
            'Date Online': row['Date Online'],
            'Owner': row['Owner'],
            'tooling_jacking': row['Jacking'] == 'Y',
            'tooling_wings': row['Wings'] == 'Y',
            'tooling_tankClosure': row['Tank Closure'] == 'Y',
            'centerline_dependencies': None if centerline_deps is None else ', '.join(centerline_deps),
            'obstructions': None,
            'notes': None
        })

    return pd.DataFrame(rows)

def build_labor_df(fgi_staffing_byshift, fgi_cpj):
    rows = []

    # FGI staffing by shift
    for shift, teams in fgi_staffing_byshift.items():
        for team, manhours in teams.items():
            rows.append({
                'category': 'FGI_STAFFING_BYSHIFT',
                'shift': shift,
                'team': team,
                'value': manhours
            })

    # FGI CPJ
    for team, cpj in fgi_cpj.items():
        rows.append({
            'category': 'FGI_CPJ',
            'shift': None,
            'team': team,
            'value': cpj
        })

    return pd.DataFrame(rows)




In [35]:

# build dataframes from FARO scorecare with milestones

def clean_FAstatus(faro_scorecard, tankClosure, p3_milestones):
    ## FARO Scorecard Cleaning
    # only take rows with valid LNs
    faro_scorecard = faro_scorecard[pd.to_numeric(faro_scorecard['LN'], errors='coerce').notna()].copy()
    # remove unnamed columns
    faro_scorecard = faro_scorecard.loc[:, ~faro_scorecard.columns.astype(str).str.contains(r'^Unnamed')]

    # split zone shakes column into two columns: completed/required
    faro_scorecard[['shakes_completed', 'shakes_req']] = faro_scorecard['Zone Shakes'].astype(str).str.split('/', expand=True)
    # only take 
    faro_scorecard['p3_milestones'] = faro_scorecard['P3 Milestones'].astype(str).str.split('/').str[0]

    # set datatypes
    faro_scorecard = (
        faro_scorecard.assign(
            LN=lambda df: pd.to_numeric(df['LN'], errors='coerce').astype(int),
            **{
                'FA RO to B1R': lambda df: pd.to_numeric(df['FA RO to B1R'], errors='coerce'),
                'Total Counters': lambda df: pd.to_numeric(df['Total Counters'], errors='coerce').fillna(0),
                'Total BTG': lambda df: pd.to_numeric(df['Total BTG'], errors='coerce').fillna(0),
                'P0 BTG': lambda df: pd.to_numeric(df['P0 BTG'], errors='coerce').fillna(0),
                'P1 BTG': lambda df: pd.to_numeric(df['P1 BTG'], errors='coerce').fillna(0),
                'P2 BTG': lambda df: pd.to_numeric(df['P2 BTG'], errors='coerce').fillna(0),
                'P3 BTG': lambda df: pd.to_numeric(df['P3 BTG'], errors='coerce').fillna(0),
                'Engines BTG': lambda df: pd.to_numeric(df['Engines BTG'], errors='coerce').fillna(0),
                'Doors BTG': lambda df: pd.to_numeric(df['Doors BTG'], errors='coerce').fillna(0),
                'Test BTG': lambda df: pd.to_numeric(df['Test BTG'], errors='coerce').fillna(0),
                'Tank Closure': lambda df: df['Tank Closure'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int),
                'Ceilings': lambda df: pd.to_numeric(df['Ceilings'], errors='coerce').fillna(0),
                'Initial Tests Run': lambda df: (
                    df['Initial Tests Run'].astype(str)
                    .str.replace('%', '', regex=False)
                    .replace('nan', 0)
                    .replace('', 0)
                    .astype(float) / 100
                ),
                'shakes_completed': lambda df: pd.to_numeric(df['shakes_completed'], errors='coerce').fillna(0).astype(int),
                'shake_req': lambda df: pd.to_numeric(df['shakes_req'], errors='coerce').fillna(0).astype(int),
                'p3_milestones': lambda df: pd.to_numeric(df['p3_milestones'], errors='coerce').fillna(0).astype(int),
            }
        )
    )

    ## TANK CLOSURE Cleaning
    tankClosure['LINE_NUMBER'] = pd.to_numeric(tankClosure['LINE_NUMBER'], errors='coerce')
    tankClosure['Complete_Jobs'] = pd.to_numeric(tankClosure['Complete_Jobs'], errors='coerce').fillna(0)
    tankClosure['Total_Jobs'] = pd.to_numeric(tankClosure['Total_Jobs'], errors='coerce').fillna(0)
    tankClosure['TankStatus'] = tankClosure['TankStatus'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int)

    ## P3 Milestone Cleaning
    p3_milestones['Engine_Type'] = p3_milestones['Milestone'].astype(str).str.extract(r'\((.*?)\)')
    p3_milestones['Milestone'] = p3_milestones['Milestone'].astype(str).str.replace(r'\s*\(.*?\)', '', regex=True)
    p3_milestones = (
        p3_milestones.pivot_table(index='P', columns='Milestone', values='Completed_Jobs', aggfunc='sum')
        .assign(
            Engine_Type=p3_milestones.groupby('P')['Engine_Type'].first(),
            Milestone_Completion_Percentage=p3_milestones.groupby('P')['STATUS (1 Complete, 0 Still Open)'].mean()
        )
        .fillna(-1)
        .reset_index()
    )

    return faro_scorecard, tankClosure, p3_milestones

def clean_nodeData(nodes, node_adj):
    ## clean nodes
    nodes = (
        nodes
        .drop(columns=[c for c in nodes.columns if str(c).startswith('Unnamed')], errors='ignore')
        .dropna(how='all')
        .assign(
            node_id=lambda df: df['node_id'].astype('string').str.strip(),
            x=lambda df: pd.to_numeric(df['x'], errors='coerce'),
            y=lambda df: pd.to_numeric(df['y'], errors='coerce'),
            type=lambda df: df['type'].astype('string').str.strip(),
            req_centerline=lambda df: df['req_centerline'].astype('string').str.strip()
        )
        .replace({'': pd.NA})
        .dropna(subset=['node_id', 'x', 'y'])
        .reset_index(drop=True)
    )

    ## clean node_adj 
    node_adj = (
        node_adj
        .drop(columns=[c for c in node_adj.columns if str(c).startswith('Unnamed')], errors='ignore')
        .dropna(how='all')
        .assign(
            from_node=lambda df: df['from_node'].astype('string').str.strip(),
            to_node=lambda df: df['to_node'].astype('string').str.strip()
        )
        .replace({'': pd.NA})
        .dropna(subset=['from_node', 'to_node'])
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return nodes, node_adj

def import_data(rootpath, filepaths = {'FAstatus': 'FA_Status_FGI_Handoff.xlsx','FGI_Locations': 'FGI_Locations_wPriority.xlsx','Nodes': 'Nodes.xlsx'}):
    
    ## FA STATUS DATA IMPORT
    path = filepaths['FAstatus']
    # read FARO scorecard as df
    faro_scorecard = pd.read_excel(path, sheet_name='FARO_Scorecard',header=2,engine='openpyxl')
    tankClosure = pd.read_excel(path,sheet_name='Tank_Closure_Detail',engine='openpyxl')
    p3_milestones = pd.read_excel(path, sheet_name='P3 Milestone Detail',engine='openpyxl')

    ## FGI LOCATION DATA IMPORT
    path = filepaths['FGI_Locations']
    fa_priority = pd.read_excel(path, sheet_name='FA Priority', header=1,engine='openpyxl')

    ## NODES DATA IMPORT
    path = filepaths['Nodes']
    nodes = pd.read_excel(path,sheet_name='Node',engine='openpyxl')
    node_adj = pd.read_excel(path, sheet_name='adjacency',engine='openpyxl')

    ## DATA CLEANING
    faro_scorecard, tankClosure, p3_milestones = clean_FAstatus(faro_scorecard, tankClosure, p3_milestones)

    fa_priority = (
        fa_priority
        .drop(columns=[c for c in fa_priority.columns if str(c).startswith('Unnamed')], errors='ignore')
        .dropna(how='all')
        .reset_index(drop=True)
    )

    nodes, node_adj = clean_nodeData(nodes, node_adj)

    return faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj

faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj = import_data(rootpath, filepaths)
ap_df = build_ap_df(faro_scorecard, p3_milestones, tankClosure)

location_df = build_location_df(
    fa_priority=fa_priority,
    centerlines=CENTERLINES
)

labor_df = build_labor_df(
    fgi_staffing_byshift=FGI_STAFFING_BYSHIFT,
    fgi_cpj=FGI_CPJ
)


staffing_df = labor_df[labor_df['category'] == 'FGI_STAFFING_BYSHIFT'].copy()
staffing_df['shift'] = staffing_df['shift'].astype(int)
staffing_df['value'] = pd.to_numeric(staffing_df['value'])

FGI_STAFFING_BYSHIFT = {
    shift: skill.set_index('team')['value'].to_dict()
    for shift, skill in staffing_df.groupby('shift', sort=True)
}

FGI_CPJ = (
    labor_df[labor_df['category'] == 'FGI_CPJ']
    .assign(value=lambda df: pd.to_numeric(df['value']))
    .set_index('team')['value']
    .to_dict()
)


move_times_df = pd.read_excel(filepaths['Move Times'], index_col=0)
move_times_df.index = move_times_df.index.astype(str)
move_times_df.columns = move_times_df.columns.astype(str)

paint_schedule_df = pd.read_excel(
    filepaths['Paint Schedule'],
    sheet_name='Historical',
    header=2,
    engine='openpyxl'
)[['Date', 'BSC1', 'BSC2']].copy()
paint_schedule_df['Date'] = pd.to_datetime(paint_schedule_df['Date'], errors='coerce').dt.normalize()
paint_schedule_df = paint_schedule_df[paint_schedule_df['Date'].notna()].reset_index(drop=True)



In [36]:
# export staged live-state workbook
LIVE_STATE_FILE.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(LIVE_STATE_FILE, engine='openpyxl') as writer:
    ap_df.to_excel(writer, sheet_name='ap_data', index=False)
    location_df.to_excel(writer, sheet_name='location_data', index=False)
    labor_df.to_excel(writer, sheet_name='labor_data', index=False)
    move_times_df.to_excel(writer, sheet_name='move_times')
    paint_schedule_df.to_excel(writer, sheet_name='paint_schedule', index=False)

wb = load_workbook(LIVE_STATE_FILE)

header_fill = PatternFill('solid', fgColor='1F4E78')
header_font = Font(color='FFFFFF', bold=True)
index_fill = PatternFill('solid', fgColor='D9EAF7')
thin_gray = Side(style='thin', color='BFBFBF')
date_headers = {'FA RO', 'Date Online', 'Date'}

for ws in wb.worksheets:
    ws.freeze_panes = 'B2'
    ws.sheet_view.showGridLines = False

    for row in ws.iter_rows():
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical='center', wrap_text=True)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    # Use the trace workbook's first-column visual cue without changing non-date data formats.
    for row_idx in range(2, ws.max_row + 1):
        cell = ws.cell(row=row_idx, column=1)
        cell.fill = index_fill
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    for col_idx in range(1, ws.max_column + 1):
        header_value = ws.cell(row=1, column=col_idx).value
        if header_value in date_headers:
            for row_idx in range(2, ws.max_row + 1):
                ws.cell(row=row_idx, column=col_idx).number_format = 'yyyy-mm-dd'

    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = 0
        col_letter = get_column_letter(col_idx)

        for cell in col_cells:
            value = cell.value
            if value is not None:
                max_len = max(max_len, len(str(value)))

        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 30)

    ws.auto_filter.ref = ws.dimensions

wb.save(LIVE_STATE_FILE)

print(LIVE_STATE_FILE)


data/staged/FGI_liveState.xlsx
